In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append("../src")

import eda_utils as eda
import preprocessing as prep
import visualization as visual
import modeling as mod
import feature_engineering as fe
import constants as const

importlib.reload(eda)
importlib.reload(prep)
importlib.reload(visual)
importlib.reload(mod)
importlib.reload(fe)

<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering
</h1>


En este notebook se aplica la versión final de feature engineering seleccionada para el modelo.

Las distintas combinaciones de features fueron evaluadas previamente en el notebook [experimenting.ipynb](experimenting.ipynb), donde se compararon variantes usando las mismas métricas de validación. A partir de esos resultados, se eligió la configuración con mejor desempeño y con una complejidad razonable.

Lo que hicimos entre estos dos notebooks fue:

- **Experimentación:** probar distintas combinaciones de features y elegir la mejor con evidencia empírica.
- **Feature engineering final:** aplicar únicamente la transformación seleccionada y guardar los datasets finales para modelado.

In [ ]:
SPLITTED_DATA_DIR = "../data/processed/splitted"

X_train = pd.read_csv(f"{SPLITTED_DATA_DIR}/X_train.csv")
X_val = pd.read_csv(f"{SPLITTED_DATA_DIR}/X_val.csv")
y_train = pd.read_csv(f"{SPLITTED_DATA_DIR}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{SPLITTED_DATA_DIR}/y_val.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Criteria
</h3>

In [ ]:
REFERENCE_YEAR = 2025
ZERO_KM_THRESHOLD = 100
PREMIUM_BRANDS = const.PREMIUM_BRANDS
BRAND_MODEL_MIN_COUNT = 20

COLUMNS_TO_DROP = [
    "Título",
    "Descripción",
    "Versión",
]

CATEGORICAL_COLS = [
    "Marca",
    "Modelo",
    "Marca_Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

BINARY_MISSING_COLS = [
    "Con cámara de retroceso",
]

ALL_FEATURE_BLOCKS = [
    "usage",
    "brand_model",
    "premium",
    "cilindrada_missing",
]


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Candidate Feature Preview
</h3>

Primero se aplican todas las features candidatas para revisar qué columnas generan y verificar que la transformación funcione correctamente. Esta celda es descriptiva: no implica que todas las features vayan a quedar en el modelo final.

In [ ]:
X_train_fe, X_val_fe = fe.add_initial_features(
    X_train,
    X_val,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
)


In [ ]:
new_features = [
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Marca_Modelo",
    "Es_Alta_Gama",
    "Cilindrada_missing",
]

X_train_fe[new_features].head()


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Selected Feature Set
</h3>

Después de correr los experimentos en `experimenting.ipynb`, completar las variables `SELECTED_FEATURE_BLOCKS` y `SELECTED_DROP_COLS` con la mejor configuración encontrada.

Por ahora se deja una configuración base editable. La intención es que este bloque sea el único lugar donde se fija la variante final.

In [ ]:
# TODO: update these values after reviewing the experiment table in experimenting.ipynb.
SELECTED_FEATURE_BLOCKS = ["premium"]
SELECTED_DROP_COLS = []

X_train_selected, X_val_selected, selected_categories_map = fe.build_feature_variant(
    X_train,
    X_val,
    feature_blocks=SELECTED_FEATURE_BLOCKS,
    drop_cols=SELECTED_DROP_COLS,
    columns_to_drop=COLUMNS_TO_DROP,
    categorical_cols=CATEGORICAL_COLS,
    binary_missing_cols=BINARY_MISSING_COLS,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
)

print(f"Selected feature blocks: {SELECTED_FEATURE_BLOCKS}")
print(f"Dropped columns: {SELECTED_DROP_COLS if SELECTED_DROP_COLS else 'none'}")
print(f"X_train_selected shape: {X_train_selected.shape}")
print(f"X_val_selected shape: {X_val_selected.shape}")

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Checking Selected Dataset
</h3>

In [ ]:
feature_types = eda.feature_types_summary(X_train_selected)
display(feature_types.head(30).style.hide(axis="index"))

In [ ]:
train_plot_data = visual.build_plot_dataset(X_train_selected, y_train)

candidate_cols = [
    "Precio",
    "Antigüedad",
    "Kilómetros_por_año",
    "Es_0km",
    "Es_Alta_Gama",
    "Cilindrada",
    "Cilindrada_missing",
]
selected_cols = [column for column in candidate_cols if column in train_plot_data.columns]

visual.plot_numeric_correlation_heatmap(
    train_plot_data,
    numeric_cols=selected_cols,
    add_log_price=False,
    include_target=False,
    include_log_target=False,
    title="Correlación de features seleccionadas con Precio",
)

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Saving Selected Features
</h3>

Cuando se confirme la variante final, se pueden guardar estos archivos para que `modeling.ipynb` los cargue directamente. Por ahora las líneas quedan comentadas para evitar sobrescribir resultados sin querer.

In [ ]:
# FEATURE_ENGINEERING_DATA_DIR = "../data/processed/feature_engineering"
# X_train_selected.to_csv(f"{FEATURE_ENGINEERING_DATA_DIR}/X_train_selected.csv", index=False)
# X_val_selected.to_csv(f"{FEATURE_ENGINEERING_DATA_DIR}/X_val_selected.csv", index=False)